# Pose Extraction using MediaPipe

Script ini melakukan ekstraksi 33 landmark pose dari video di dalam dataset secara rekursif. 
Hasil ekstraksi disimpan ke dalam file `.csv`.

**Jika Anda mendapatkan 'AttributeError: module mediapipe has no attribute solutions', jalankan sel instalasi di bawah ini.**

In [2]:
# Perbaikan: Pastikan library terinstall dengan benar di environment kernel ini
import sys
!{sys.executable} -m pip install mediapipe opencv-python pandas tqdm --upgrade

^C


In [1]:
import cv2
import pandas as pd
import os
from pathlib import Path
from tqdm import tqdm

# Gunakan import langsung untuk menghindari masalah namespace pada beberapa versi atau environment
try:
    import mediapipe as mp
    if not hasattr(mp, 'solutions'):
        raise AttributeError
    mp_pose = mp.solutions.pose
    print("MediaPipe loaded successfully using standard import.")
except:
    print("Standard import gagal. Mencoba memuat langsung dari submodule...")
    try:
        import mediapipe.python.solutions.pose as mp_pose
        print("MediaPipe loaded successfully using submodule import.")
    except Exception as e:
        print(f"MediaPipe gagal dimuat: {e}")
        print("Saran: Pastikan tidak ada file bernama 'mediapipe.py' di folder ini dan restart kernel.")
        raise e

# Inisialisasi Pose
pose = mp_pose.Pose(
    static_image_mode=False,
    model_complexity=1,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

def extract_pose_landmarks(dataset_path, output_csv):
    dataset_dir = Path(dataset_path)
    video_files = list(dataset_dir.rglob('*.mp4'))
    
    data = []
    print(f"Ditemukan {len(video_files)} file video.")
    
    for video_path in tqdm(video_files, desc="Memproses Video"):
        # Tentukan label
        parts = video_path.parts
        
        main_label = "Unknown"
        if "fokus" in parts:
            main_label = "fokus"
        elif "tidak_fokus" in parts:
            main_label = "tidak_fokus"
            
        sub_label = video_path.parent.name
        video_id = video_path.name
        
        cap = cv2.VideoCapture(str(video_path))
        frame_num = 0
        
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
            
            frame_num += 1
            # Konversi BGR ke RGB
            image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            results = pose.process(image_rgb)
            
            # Baris data dasar
            row = [video_id, frame_num, main_label, sub_label]
            
            if results.pose_landmarks:
                for res in results.pose_landmarks.landmark:
                    row.extend([res.x, res.y, res.z, res.visibility])
            else:
                # Zero-padding
                row.extend([0.0] * (33 * 4))
            
            data.append(row)
            
        cap.release()
    
    # Membuat daftar kolom
    columns = ['video_id', 'frame_num', 'main_label', 'sub_label']
    for i in range(33):
        columns.extend([f'x_{i}', f'y_{i}', f'z_{i}', f'v_{i}'])
    
    df = pd.DataFrame(data, columns=columns)
    df.to_csv(output_csv, index=False)
    print(f"\nProses selesai. Data disimpan ke {output_csv}")
    return df

Standard import gagal. Mencoba memuat langsung dari submodule...
MediaPipe gagal dimuat: No module named 'mediapipe.python'
Saran: Pastikan tidak ada file bernama 'mediapipe.py' di folder ini dan restart kernel.


ModuleNotFoundError: No module named 'mediapipe.python'

In [ ]:
# Jalankan ekstraksi
DATASET_PATH = '.' # Sesuaikan jika path berbeda, '.' adalah folder saat ini
OUTPUT_FILE = 'ekstraksi_pose_lengkap.csv'

# Karena script ini biasanya ada dalam folder Dataset, kita gunaakan '.'
df_result = extract_pose_landmarks(DATASET_PATH, OUTPUT_FILE)

  Using cached mediapipe-0.10.33-py3-none-win_amd64.whl.metadata (9.8 kB)
  Using cached opencv_python-4.13.0.92-cp37-abi3-win_amd64.whl.metadata (20 kB)
Using cached mediapipe-0.10.33-py3-none-win_amd64.whl (10.6 MB)
Using cached opencv_python-4.13.0.92-cp37-abi3-win_amd64.whl (40.2 MB)

  Attempting uninstall: opencv-python

    Found existing installation: opencv-python 4.11.0.86

    Uninstalling opencv-python-4.11.0.86:

   ---------------------------------------- 0/2 [opencv-python]



ERROR: Could not install packages due to an OSError: [WinError 5] Access is denied: 'd:\\programs\\anaconda3\\envs\\yolov11\\lib\\site-packages\\cv2\\cv2.pyd'
Consider using the `--user` option or check the permissions.



In [ ]:
import mediapipe as mp
import os
print(f"Versi MediaPipe: {mp.__version__}")
print(f"Lokasi File: {mp.__file__}")
print(f"Daftar Atribut: {dir(mp)}")


Versi MediaPipe: 0.10.32
Lokasi File: d:\Programs\anaconda3\envs\yolov11\Lib\site-packages\mediapipe\__init__.py
Daftar Atribut: ['Image', 'ImageFormat', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '__version__', 'tasks']


In [2]:
import sys
print(sys.executable)

d:\Programs\anaconda3\envs\yolov11\python.exe
